# Phase 9.1-9.2: Power Analysis

Power analysis for sample size and parameter variation effects on Markovianity diagnostic performance.

## Workflow
1. Define reduced grid: T [500,1000,2000], d [5,10,20], 10 repeats
2. Run power analysis with latent confounder scenario
3. Compute metrics: TPR, FPR, selected p_star, runtime
4. Aggregate over repeats
5. Export and visualize results


### Setup: Imports and Project Root


In [ ]:
from __future__ import annotations

import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project setup
root = Path('/Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics')
sys.path.insert(0, str(root / 'src'))

from markovianity_diagnostic.experiments.power import (
    PowerAnalyzer,
    PowerGrid,
)

# Plotting setup
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## Power Grid Definition

Reduced grid for notebook execution:
- T (sample sizes): [500, 1000, 2000]
- d (dimensions): [5, 10, 20]
- Repeats per grid point: 10


In [ ]:
# Create power grid (reduced for notebook)
grid = PowerGrid(
    T_values=[500, 1000, 2000],
    d_values=[5, 10, 20],
)

print(f'Grid size: {grid.total_combinations()} combinations')
print(f'With 10 repeats: {grid.total_combinations() * 10} total runs')


## Run Power Analysis

Execute power analysis across the grid with 10 repeats per point.
This measures how TPR/FPR/p_star vary with sample size and dimension.


In [ ]:
# Initialize analyzer
analyzer = PowerAnalyzer(
    grid=grid,
    method='gcstar_cgc',
    repeats=10,
    p_values=[1, 2, 3, 4, 5, 6],
    verbose=True,
    seed=42,
)

print('Starting power analysis...')
start_time = datetime.now(timezone.utc)

results = analyzer.run()

end_time = datetime.now(timezone.utc)
elapsed = (end_time - start_time).total_seconds()

print(f'\nCompleted {len(results)} runs in {elapsed:.1f} seconds')
print(f'Average time per run: {elapsed / len(results):.2f} seconds')


## Aggregate Results

Compute mean and standard deviation of metrics over repeats for each grid point.


In [ ]:
# Aggregate results
aggregated = analyzer.aggregate(results)

print(f'Aggregated {len(aggregated)} grid points')
print('\nSample aggregation:')
print(aggregated[0])


## Export Results

Export detailed and aggregated results to output directory.


In [ ]:
# Create output directory
output_dir = root / 'outputs' / 'power' / 'sample_size_power'
output_dir.mkdir(parents=True, exist_ok=True)

# Export
analyzer.export(results, aggregated, output_dir)

print(f'Exported to {output_dir}')
print(f'Files:')
for f in sorted(output_dir.glob('*')):
    print(f'  {f.name}')


## Load Summary for Analysis


In [ ]:
# Load aggregated results into DataFrame
summary_csv = output_dir / 'summary.csv'
df = pd.read_csv(summary_csv)

print(f'Loaded {len(df)} rows')
print('\nColumn summary:')
print(df.dtypes)
print('\nFirst few rows:')
df.head()


## TPR vs Sample Size

How does True Positive Rate improve with sample size?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, d in enumerate(sorted(df['d'].unique())):
    ax = axes[i]
    subset = df[df['d'] == d]
    
    for T in sorted(subset['T'].unique()):
        T_data = subset[subset['T'] == T]
        ax.scatter(T_data['T'], T_data['mean_tpr'], label=f'T={T}', s=100, alpha=0.7)
    
    ax.set_xlabel('Sample Size T')
    ax.set_ylabel('Mean TPR')
    ax.set_title(f'd={d}')
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
power_curve_path = output_dir / 'tpr_by_sample_size.png'
plt.savefig(power_curve_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved: {power_curve_path}')


## TPR vs Dimension

How does dimension affect performance?


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for T in sorted(df['T'].unique()):
    subset = df[df['T'] == T]
    ax.plot(subset['d'], subset['mean_tpr'], marker='o', label=f'T={T}', linewidth=2)
    ax.fill_between(
        subset['d'],
        subset['mean_tpr'] - subset['std_tpr'],
        subset['mean_tpr'] + subset['std_tpr'],
        alpha=0.2,
    )

ax.set_xlabel('Dimension d')
ax.set_ylabel('Mean TPR')
ax.set_title('TPR vs Dimension (with std error bands)')
ax.set_ylim([0, 1])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
tpr_dim_path = output_dir / 'tpr_by_dimension.png'
plt.savefig(tpr_dim_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved: {tpr_dim_path}')


## False Positive Rate Analysis


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for T in sorted(df['T'].unique()):
    subset = df[df['T'] == T]
    ax.plot(subset['d'], subset['mean_fpr'], marker='s', label=f'T={T}', linewidth=2)
    ax.fill_between(
        subset['d'],
        subset['mean_fpr'] - subset['std_fpr'],
        subset['mean_fpr'] + subset['std_fpr'],
        alpha=0.2,
    )

ax.set_xlabel('Dimension d')
ax.set_ylabel('Mean FPR')
ax.set_title('False Positive Rate vs Dimension')
ax.set_ylim([0, 1])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fpr_path = output_dir / 'fpr_by_dimension.png'
plt.savefig(fpr_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved: {fpr_path}')


## Selected Depth (p_star) Analysis

How does the selected conditioning depth vary with sample size?


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for T in sorted(df['T'].unique()):
    subset = df[df['T'] == T]
    ax.plot(subset['d'], subset['mean_p_star'], marker='^', label=f'T={T}', linewidth=2)

ax.set_xlabel('Dimension d')
ax.set_ylabel('Mean p_star')
ax.set_title('Selected Depth vs Dimension')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(sorted(df['d'].unique()))

plt.tight_layout()
pstar_path = output_dir / 'pstar_by_dimension.png'
plt.savefig(pstar_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved: {pstar_path}')


## Runtime Scaling

How does computational time scale?


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for d in sorted(df['d'].unique()):
    subset = df[df['d'] == d]
    ax.plot(subset['T'], subset['mean_runtime'], marker='o', label=f'd={d}', linewidth=2)

ax.set_xlabel('Sample Size T')
ax.set_ylabel('Mean Runtime (seconds)')
ax.set_title('Runtime vs Sample Size')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
runtime_path = output_dir / 'runtime_scaling.png'
plt.savefig(runtime_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved: {runtime_path}')


## Summary Statistics


In [ ]:
print('\n=== Power Analysis Summary ===')
print(f'\nTotal grid points: {len(aggregated)}')
print(f'Repeats per point: 10')
print(f'Total runs: {len(results)}')

print('\n=== TPR Statistics ===')
print(f'Mean TPR: {df["mean_tpr"].mean():.3f}')
print(f'Min TPR: {df["mean_tpr"].min():.3f}')
print(f'Max TPR: {df["mean_tpr"].max():.3f}')

print('\n=== FPR Statistics ===')
print(f'Mean FPR: {df["mean_fpr"].mean():.3f}')
print(f'Min FPR: {df["mean_fpr"].min():.3f}')
print(f'Max FPR: {df["mean_fpr"].max():.3f}')

print('\n=== p_star Statistics ===')
print(f'Mean p_star: {df["mean_p_star"].mean():.2f}')
print(f'Min p_star: {df["mean_p_star"].min():.2f}')
print(f'Max p_star: {df["mean_p_star"].max():.2f}')

print('\n=== Runtime Statistics ===')
print(f'Mean runtime: {df["mean_runtime"].mean():.2f}s')
print(f'Min runtime: {df["mean_runtime"].min():.2f}s')
print(f'Max runtime: {df["mean_runtime"].max():.2f}s')
